In [0]:
# --- Notebook: 03_Gold_Commuter_Analysis ---
from pyspark.sql.functions import col, when, hour, dayofweek

# 1. Load Silver Tables
# We use the SCD2 Dim table to get the "correct" names/locations for each trip
df_trips = spark.table("main.citibike_lakehouse.silver_trips")
df_stations = spark.table("main.citibike_lakehouse.dim_stations")

# 2. Join Fact to Dimension (Point-in-Time lookup)
# We join where the trip 'started_at' falls between the station's 'start_date' and 'end_date'
fact_trips = df_trips.join(
    df_stations,
    (df_trips.start_station_id == df_stations.station_id) & 
    (df_trips.started_at >= df_stations.start_date) & 
    (df_trips.started_at <= df_stations.end_date),
    "left"
).select(
    df_trips["*"], 
    df_stations["station_name"].alias("resolved_station_name")
)

# 3. Commuter Logic: Identifying Rush Hour Patterns
# Defining Rush Hours: 7-10 AM and 4-7 PM on Weekdays
gold_commuter_df = (fact_trips
    .withColumn("is_rush_hour", 
        (col("day_of_week").between(2, 6)) & # Monday to Friday
        ((col("hour_of_day").between(7, 10)) | (col("hour_of_day").between(16, 19)))
    )
    .withColumn("commuter_segment", 
        when(col("is_rush_hour") & (col("is_member") == True), "Subscriber_Commuter")
        .when(col("is_rush_hour") & (col("is_member") == False), "Casual_Commuter")
        .otherwise("Leisure_Rider"))
)

# 4. Aggregation: Top Commuter Routes
# This is the "Value Add" table for your BI Dashboard
gold_route_analysis = (gold_commuter_df
    .filter(col("commuter_segment") != "Leisure_Rider")
    .groupBy("start_station_name", "end_station_name", "commuter_segment", "bike_category")
    .agg(
        expr("count(*) as total_trips"),
        expr("round(avg(duration_sec)/60, 2) as avg_duration_min"),
        expr("round(avg(distance_km), 2) as avg_distance_km")
    )
    .orderBy(col("total_trips").desc())
)

# 5. Write to Gold & Apply Z-Ordering
(gold_route_analysis.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("main.citibike_lakehouse.gold_commuter_routes"))

# 6. Performance Optimization
spark.sql("OPTIMIZE main.citibike_lakehouse.gold_commuter_routes ZORDER BY (start_station_name)")